In [1]:
print("hello")

hello


In [5]:
pip install transformers peft trl accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 19.7 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
# dependency issue -- `torch_dtype` is deprecated! Use `dtype` instead!
!pip install bitsandbytes>=0.46.1

In [7]:
!pip install -U trl==0.12.0 --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 7.8 MB/s eta 0:00:00:00:01
  Attempting uninstall: trl
    Found existing installation: trl 1.8.0
    Uninstalling trl-1.8.0:
      Successfully uninstalled trl-1.8.0


# DATASET ALTERATION FOR SFT NOW

In [2]:
import json
from pathlib import Path
from collections import defaultdict
import random


INPUT_PATH = "/kaggle/input/datasets/ibtihussain/pref-pair-sft-dpo/preference_pairs.jsonl"
TRAIN_OUTPUT_PATH = "/kaggle/working/sft_train.jsonl"
HOLDOUT_OUTPUT_PATH = "/kaggle/working/sft_holdout.jsonl"  

SEED = 10
HOLDOUT_FRACTION = 0.2 # 20% MEANS 6 OUT OF 30 TASKS IDs WILL BE FOR EVALS


all_pairs = []
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        all_pairs.append(json.loads(line))

print(f"Loaded {len(all_pairs)} total preference pairs")

Loaded 112 total preference pairs


In [3]:
# --- Split by task_id (not by row) to avoid leakage across train/eval ---
unique_task_ids = sorted(set(p["task_id"] for p in all_pairs))
print(f"Unique task_ids: {len(unique_task_ids)}")

random.seed(SEED)
shuffled_task_ids = unique_task_ids.copy()
random.shuffle(shuffled_task_ids)

num_holdout = max(1, round(len(shuffled_task_ids) * HOLDOUT_FRACTION))
holdout_task_ids = set(shuffled_task_ids[:num_holdout])
train_task_ids = set(shuffled_task_ids[num_holdout:])

print(f"Holdout task_ids ({len(holdout_task_ids)}): {sorted(holdout_task_ids)}")
print(f"Train task_ids ({len(train_task_ids)}): {sorted(train_task_ids)}")

Unique task_ids: 30
Holdout task_ids (6): ['task_05', 'task_08', 'task_10', 'task_18', 'task_21', 'task_23']
Train task_ids (24): ['task_01', 'task_02', 'task_03', 'task_04', 'task_06', 'task_07', 'task_09', 'task_11', 'task_12', 'task_13', 'task_14', 'task_15', 'task_16', 'task_17', 'task_19', 'task_20', 'task_22', 'task_24', 'task_25', 'task_26', 'task_27', 'task_28', 'task_29', 'task_30']


In [4]:
# Build SFT examples: prompt + chosen only (drop rejected for this stage)
def build_sft_record(pair):
    return {
        "task_id": pair["task_id"],
        "tool": pair["tool"],
        "tier": pair["tier"],
        "messages": [
            {"role": "user", "content": pair["prompt"]},
            {"role": "assistant", "content": pair["chosen"]},
        ],
    }

train_records = [build_sft_record(p) for p in all_pairs if p["task_id"] in train_task_ids]
holdout_records = [build_sft_record(p) for p in all_pairs if p["task_id"] in holdout_task_ids]

print(f"\nSFT train examples: {len(train_records)}")
print(f"SFT holdout examples: {len(holdout_records)}")



SFT train examples: 92
SFT holdout examples: 20


In [5]:
with open(TRAIN_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for rec in train_records:
        f.write(json.dumps(rec) + "\n")

with open(HOLDOUT_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for rec in holdout_records:
        f.write(json.dumps(rec) + "\n")

print(f"\nSaved train set to {TRAIN_OUTPUT_PATH}")
print(f"Saved holdout set to {HOLDOUT_OUTPUT_PATH}")


Saved train set to /kaggle/working/sft_train.jsonl
Saved holdout set to /kaggle/working/sft_holdout.jsonl


# LLM for fine tuning

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-3B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

# --- 4-bit quantization config (QLoRA) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(model)
print(f"\nModel loaded. Vocab size: {len(tokenizer)}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

In [2]:
# Prepare model for k-bit training and attach LoRA adapters
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [3]:
from datasets import Dataset
import json

SFT_TRAIN_PATH = "/kaggle/working/sft_train.jsonl"

# Load the JSONL into a list of dicts 
train_data = []
with open(SFT_TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        train_data.append(json.loads(line))

print(f"Loaded {len(train_data)} SFT training examples")

# Qwen's chat template to format each example as a single training string 
def format_example(example):
    formatted_text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": formatted_text}

formatted_data = [format_example(ex) for ex in train_data]

# token length distribution before committing to a max_seq_length
lengths = [len(tokenizer.encode(ex["text"])) for ex in formatted_data]
print(f"\nToken length stats:")
print(f"  min: {min(lengths)}")
print(f"  max: {max(lengths)}")
print(f"  mean: {sum(lengths)/len(lengths):.0f}")
print(f"  95th percentile: {sorted(lengths)[int(len(lengths)*0.95)]}")

# Build the HF Dataset
train_dataset = Dataset.from_list(formatted_data)
print(f"\nDataset ready: {train_dataset}")
print(f"\nExample formatted text:\n{'-'*50}\n{train_dataset[0]['text'][:800]}")

Loaded 92 SFT training examples

Token length stats:
  min: 143
  max: 568
  mean: 275
  95th percentile: 512

Dataset ready: Dataset({
    features: ['text'],
    num_rows: 92
})

Example formatted text:
--------------------------------------------------
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Review this code:

def sum_numeric_strings(numbers):
    return sum(float(n) for n in numbers)<|im_end|>
<|im_start|>assistant
The function correctly uses a generator expression with `sum()` and `float()` for a clean, Pythonic approach to numeric conversion. However, it lacks defensive programming for two critical edge cases: (1) the function crashes when `numbers` is `None` instead of handling it gracefully—add a guard clause like `if numbers is None: return 0` at the start; (2) it fails on malformed input like 'abc' without validation—wrap the `float(n)` conversion in a try-except block to either skip invalid entries or

In [5]:
from trl import SFTTrainer, SFTConfig

MAX_SEQ_LENGTH = 800  # comfortably covers your 568 max with headroom

sft_config = SFTConfig(
    output_dir="/kaggle/working/aicr_sft_checkpoint",
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch size = 8
    num_train_epochs=3,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()

Map:   0%|          | 0/92 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and br

Step,Training Loss
5,1.618177
10,1.171266
15,0.856877
20,0.692255
25,0.620855
30,0.561795
35,0.547050


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=36, training_loss=0.8576819366878934, metrics={'train_runtime': 1050.5882, 'train_samples_per_second': 0.263, 'train_steps_per_second': 0.034, 'total_flos': 1526691949264896.0, 'train_loss': 0.8576819366878934, 'epoch': 3.0})

In [6]:
SFT_ADAPTER_PATH = "/kaggle/working/aicr_sft_adapter"

trainer.model.save_pretrained(SFT_ADAPTER_PATH)
tokenizer.save_pretrained(SFT_ADAPTER_PATH)

print(f"SFT adapter saved to {SFT_ADAPTER_PATH}")

SFT adapter saved to /kaggle/working/aicr_sft_adapter


# Fined tuned Qwen's inference sanity check 

In [8]:
from peft import PeftModel
import torch

SFT_ADAPTER_PATH = "/kaggle/working/aicr_sft_adapter"


sft_model = PeftModel.from_pretrained(model, SFT_ADAPTER_PATH)
sft_model.eval()

sft_tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_PATH)

# --- Load a couple of held-out examples to test on ---
holdout_examples = []
with open("/kaggle/working/sft_holdout.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        holdout_examples.append(json.loads(line))

print(f"Loaded {len(holdout_examples)} held-out examples")



/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.o_proj.

Loaded 20 held-out examples


In [10]:
# --- Picked 3 examples from different held-out task_ids ---
seen_tasks = set()
sample_examples = []
for ex in holdout_examples:
    if ex["task_id"] not in seen_tasks:
        sample_examples.append(ex)
        seen_tasks.add(ex["task_id"])
    if len(sample_examples) == 3:
        break

# --- Run inference on each sample ---
def generate_review(model, tokenizer, prompt_text, max_new_tokens=700):
    messages = [{"role": "user", "content": prompt_text}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


for i, ex in enumerate(sample_examples):
    prompt_text = ex["messages"][0]["content"]
    ground_truth_chosen = ex["messages"][1]["content"]

    print(f"\n{'='*70}")
    print(f"HELD-OUT EXAMPLE {i+1} — task_id: {ex['task_id']}, tool: {ex['tool']}, tier: {ex['tier']}")
    print(f"{'='*70}")
    print(f"\n--- PROMPT (snippet) ---\n{prompt_text[:500]}...")
    print(f"\n--- SFT MODEL OUTPUT ---\n{generate_review(sft_model, sft_tokenizer, prompt_text)}")
    print(f"\n--- REFERENCE (chosen, from your dataset) ---\n{ground_truth_chosen}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



HELD-OUT EXAMPLE 1 — task_id: task_05, tool: gpt, tier: 1

--- PROMPT (snippet) ---
Review this code:

from collections.abc import Mapping


def flatten_json(data, parent_key="", sep="."):
    """
    Flatten a nested JSON-like dictionary into a flat dictionary
    using dot-notation keys.

    Example:
        {"a": {"b": 1}, "c": 2}
        -> {"a.b": 1, "c": 2}
    """
    flat = {}

    for key, value in data.items():
        new_key = f"{parent_key}{sep}{key}" if parent_key else str(key)

        if isinstance(value, Mapping):
            flat.update(flatten_json(value, ne...

--- SFT MODEL OUTPUT ---
The provided code snippet is designed to flatten a nested JSON-like dictionary into a single-level dictionary using dot-notation keys. Here's a review of the code:

### Function Definition and Docstring
- **Function Name**: `flatten_json`
- **Parameters**:
  - `data`: The input dictionary to be flattened.
  - `parent_key`: A string representing the current path in the dictionary. De

SFT QWEN ADAPTERS NOT BEHAIVING WELL ACCORDING TO THE DATASET, NOT TRAINED WELL ENOUGH 


* Test greedy decoding + longer output (free, no retraining) — rule this out first
* Increase epochs (5-8) — free, just retrain longer on the same data
* Increase LoRA rank (16→32) — free, just retrain with a bigger adapter
* Strengthen the instruction in the prompt itself (e.g., explicitly say "review with validation then critique" rather than just "review this code") — free, just changes prompt phrasing
* Only after 1-4 — if the behavior still doesn't shift, then genuinely consider whether you need more/better data (e.g., expanding Tier 1 rejected variants, or adding more distinct task examples)